# Round 15 — Action-centered and qualification-centered evidence windows

One bounded feature experiment. No neural inference, downloads or ensemble search. The existing development cohort is exploratory, not a fresh holdout or Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/action_window_features.json").is_file())
from scripts.run_action_window_features import run_study, figures, write_dashboard
config = json.loads((ROOT / "configs/action_window_features.json").read_text())
print("Primary:",config["primary"],"| New fits:",config["new_fits"])
print("Candidate feature columns:",config["new_primary_columns"])


Primary: window_all | New fits: 12
Candidate feature columns: 48


## Evidence and fixed hypothesis

Windows surround a fixed set of action and qualification cues with a twelve-token radius. Up to six merged windows retain evenly spaced positions when needed. No-cue comments use the whole comment. Displaced windows match count and token lengths, but not their lexical norms; whole-comment controls and explicit coverage report that limitation. Unselected text is deliberately omitted, not claimed preserved. These are deterministic cues, not learned rationales or true syntactic negation scopes.

No companion-round results are read.

In [2]:
for n,slug in [(12,"reference_consistency_features"),(13,"passage_support_features")]:
    prior=json.loads((ROOT / "reports" / slug / "results.json").read_text())
    print("Round",n,"decision:",prior["decision"])
    display(pd.DataFrame(prior["pooled_metrics"]))


Round 12 decision: DO_NOT_PROMOTE_PRIMARY


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,agreement_features,0.727830,0.734316,0.738909
6,reciprocity_features,0.726878,0.732264,0.738835
7,consistency_all,0.726598,0.730519,0.736879
8,quality_null_all,0.723669,0.709782,0.738809
9,geometry_only_all,0.719054,0.724195,0.730477


Round 13 decision: DO_NOT_PROMOTE_PRIMARY


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,word_passages,0.730255,0.738013,0.744573
6,character_passages,0.724681,0.734648,0.738145
7,passage_all,0.724384,0.735518,0.739711
8,whole_comment_all,0.726949,0.739874,0.742511
9,boundary_null_all,0.723435,0.733203,0.737058


## Verified compute or completed-result reuse

Use the terminal helper first. It bounds the scientific process before executing this notebook. This cell only reads its verified result and never starts a new fit.

In [3]:
result=json.loads((ROOT / "reports/action_window_features/results.json").read_text())
print("Run:",result["run_id"],"Decision:",result["decision"])
print("Cached control parity:",result["control_design_parity"])
CHARTS=figures(result)


Run: 77dd5f113ba9802ba868 Decision: DO_NOT_PROMOTE_PRIMARY
Cached control parity: True


## Per-policy performance and uncertainty

A higher mean cannot conceal a policy regression. Intervals condition on fixed predictions and do not cover all earlier adaptive choices.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## Mechanism controls and family ablations

The primary cannot be swapped for a favorable secondary candidate.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,action_windows,context_evidence,action_windows vs context anchor,0.000662,-0.018999,0.020324
1,qualification_windows,context_evidence,qualification_windows vs context anchor,-0.002686,-0.022348,0.016976
2,window_all,context_evidence,window_all vs context anchor,-0.001399,-0.021061,0.018263
3,displaced_all,context_evidence,displaced_all vs context anchor,-0.001674,-0.021336,0.017987
4,whole_comment_all,context_evidence,whole_comment_all vs context anchor,0.001025,-0.018637,0.020687
5,label_null_all,context_evidence,label_null_all vs context anchor,-0.008569,-0.028230,0.011093
6,window_all,qwen_raw,Primary vs qwen_raw,0.007964,-0.011698,0.027626
7,window_all,frozen_basic,Primary vs frozen_basic,0.004790,-0.014872,0.024452
8,window_all,uniform_all,Primary vs uniform_all,-0.001053,-0.020714,0.018609
9,window_all,displaced_all,Primary vs displaced_all,0.000275,-0.019387,0.019937


## Feature coverage and explicit missing evidence

Zero lexical overlap has zero similarity, not an invented label. Inspect coverage before interpreting averages.

In [6]:
display(pd.DataFrame(result["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,mode,family,reference_rows,vocabulary_columns,mean_windows,cue_present_fraction,selected_token_fraction,zero_lexical_fraction,new_span_labels,displaced_changed_fraction
0,0,0,0,windows,action,419,6000,1.004762,0.152381,0.963522,0.0,0,0.061905
1,0,0,0,displaced,action,419,6000,1.004762,0.152381,0.963522,0.0,0,0.061905
2,0,0,0,whole_comment,action,419,6000,1.000000,0.152381,0.963522,0.0,0,0.061905
3,0,0,0,label_null,action,419,6000,1.004762,0.152381,0.963522,0.0,0,0.061905
4,0,0,0,windows,qualification,419,6000,1.004762,0.185714,0.967042,0.0,0,0.071429
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,outer_query,0,label_null,action,366,6000,1.012365,0.174652,0.925713,0.0,0,0.143740
60,1,outer_query,0,windows,qualification,366,6000,1.103555,0.697063,0.832157,0.0,0,0.466770
61,1,outer_query,0,displaced,qualification,366,6000,1.103555,0.697063,0.832157,0.0,0,0.466770
62,1,outer_query,0,whole_comment,qualification,366,6000,1.000000,0.697063,0.832157,0.0,0,0.466770


## Associations and probability diagnostics

Coefficients are fitted associations, not causal explanations. AUC improvement does not guarantee log-loss improvement.

In [7]:
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

## Fixed decision, provenance and next gate

A pass is eligibility for later independent validation, not model promotion. Feature research remains open.

In [8]:
print("Decision:",result["decision"])
for limitation in result["limitations"]:
    print(limitation)
print("Dashboard:",write_dashboard(ROOT,result))
print("GitHub is not changed by this notebook.")

Decision: DO_NOT_PROMOTE_PRIMARY
This repeatedly inspected development cohort is not independent confirmation.
The unpromoted Round 9 context-evidence anchor was chosen adaptively before these rounds.
New reference statistics are group-cross-fitted; inherited adapted answer training margins remain in-sample.
No semantic counterpart labels or passage labels are invented; original supplied reference labels only.
Simultaneous intervals cover this round, not the entire adaptive project search.
Both Rounds 14 and 15 were specified before running either. Round 14 is not an input.
Windows surround a fixed set of action and qualification cues with a twelve-token radius. Up to six merged windows retain evenly spaced positions when needed. No-cue comments use the whole comment. Displaced windows match count and token lengths, but not their lexical norms; whole-comment controls and explicit coverage report that limitation. Unselected text is deliberately omitted, not claimed preserved. These are 